# threshold_v2 — validation selection and guarded test evaluation
This notebook never retrains models. Cells 1–7 perform validation-only calibration and policy selection. Cell 8 keeps the untouched test locked unless explicitly enabled after reviewing the validation report.

In [ ]:
# Cell 1 — Mount persistent Drive.
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cell 2 — Configure paths and REQUIRED assumptions.
REPO_URL = 'https://github.com/Qauntify/qauntify_webV1.git'
REPO_COMMIT = 'd6cf9bf'
REPO_DIR = '/content/qauntify_webV1'
DRIVE_ROOT = '/content/drive/MyDrive/Quantify/training_v1_full_001'
DATASET_ROOT = f'{DRIVE_ROOT}/datasets/datasets/training_v1'
TUNING_ROOT = f'{DRIVE_ROOT}/tuning/tuning_v1'
OUTPUT_ROOT = f'{DRIVE_ROOT}/thresholding/threshold_v2'
MINIMUM_COUNT_PER_FOLD = 100
TRADING_COST_R = 0.02

In [ ]:
# Cell 3 — Obtain the pinned repository revision.
import pathlib, subprocess, sys
if not pathlib.Path(REPO_DIR, '.git').is_dir():
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin'], check=True)
subprocess.run(['git', '-C', REPO_DIR, 'checkout', '--detach', REPO_COMMIT], check=True)

In [ ]:
# Cell 4 — Install the shared requirements.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', f'{REPO_DIR}/requirements-training.txt'], check=True)

In [ ]:
# Cell 5 — Report missing assumptions. This never reads test rows.
subprocess.run([sys.executable, '-m', 'ml.thresholding.cli', 'audit', '--config', 'ml/configs/threshold_v2.yaml'], cwd=REPO_DIR, check=True)

In [ ]:
# Cell 6 — Validation-only calibration, policy search, and optional locking.
assert isinstance(MINIMUM_COUNT_PER_FOLD, int) and MINIMUM_COUNT_PER_FOLD > 0
assert TRADING_COST_R is not None and TRADING_COST_R >= 0
subprocess.run([sys.executable, '-m', 'ml.thresholding.cli', 'select', '--config', 'ml/configs/threshold_v2.yaml', '--dataset-root', DATASET_ROOT, '--tuning-root', TUNING_ROOT, '--output-dir', OUTPUT_ROOT, '--minimum-count-per-fold', str(MINIMUM_COUNT_PER_FOLD), '--trading-cost-r', str(TRADING_COST_R)], cwd=REPO_DIR, check=True)

In [ ]:
# Cell 7 — Review this report before any test evaluation.
print(pathlib.Path(OUTPUT_ROOT, 'threshold_v2_report.md').read_text())

In [ ]:
# Cell 8 — EXACTLY-ONCE TEST GATE. Leave False until the locked validation policy is approved.
RUN_UNTOUCHED_TEST = False
assert RUN_UNTOUCHED_TEST, 'Untouched test remains locked. Review Cell 7 first.'
subprocess.run([sys.executable, '-m', 'ml.thresholding.cli', 'evaluate-test', '--config', 'ml/configs/threshold_v2.yaml', '--dataset-root', DATASET_ROOT, '--tuning-root', TUNING_ROOT, '--output-dir', OUTPUT_ROOT, '--minimum-count-per-fold', str(MINIMUM_COUNT_PER_FOLD), '--trading-cost-r', str(TRADING_COST_R), '--confirm-untouched-test'], cwd=REPO_DIR, check=True)